# 🪐 Kepler-64 Credibility Gate & Sovereign Self-Play Benchmark — Kaggle GPU

Runs the full self-play harvest → differentiable physics training → credibility match → Stockfish mock test pipeline on a free Tesla T4 GPU.

### 🌌 Philosophy: Sovereign Physics, Earthly Mirror
- **Sovereign Teacher:** Kepler-64 learns exclusively from its own deep search (1000ms gravitational simulation), preserving physical invariants and sharp forces.
- **Earthly Mirror (Mock Test):** After training, Kepler-64 sits for a 10-game mock exam against calibrated Stockfish (1400 Elo) to benchmark real-world playing strength without contaminating the loss function.

### 💤 How to run unattended (Close laptop & sleep):
1. In the top-right corner of the Kaggle notebook, click **Save Version**.
2. Under **Version Type**, select **Save & Run All (Commit)**.
3. Click **Save**.
4. A notification banner will appear at the bottom: *"Version running in background..."*.
5. **You can immediately close your browser, shut down your machine, and sleep.**
6. Kaggle runs the entire pipeline in the cloud (up to 12 hours). When you return, all checkpoints,
   logs, and markdown reports will be under the **Output** tab.


In [ ]:
# Cell 1 — dependencies, calibrated external master engine, and GPU accelerator check
!apt-get update -qq && apt-get install -y -qq stockfish > /dev/null 2>&1
!ln -sf /usr/games/stockfish /usr/local/bin/stockfish 2>/dev/null || true
%pip install -q optax python-chess
import os, shutil
os.environ['PATH'] = '/usr/games:' + os.environ.get('PATH', '')
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.20'
import jax
devices = jax.devices()
print('JAX Devices:', devices)
sf_path = shutil.which('stockfish') or ('/usr/games/stockfish' if os.path.isfile('/usr/games/stockfish') else None)
print('Stockfish binary:', sf_path)
assert any(d.platform == 'gpu' for d in devices), 'GPU not detected! Turn on GPU in Kaggle: Settings -> Accelerator -> GPU T4'


In [ ]:
# Cell 2 â€” repo (public clone; if private, upload as a Kaggle dataset instead)
!git clone -q https://github.com/r-baruah/kepler-64.git
%cd kepler-64
!python -c "import kepler64; print('import ok')"

### Batch Size Scaling & Multi-Worker Acceleration on Kaggle T4
- **Parallel Match Execution:** `--workers 4` runs head-to-head match games concurrently across all 4 vCPUs,
  slashing the 200-game match phase from ~25 minutes down to ~6 minutes.
- **Tensor Core Mini-batches:** `--batch-size 256` keeps GPU tensor cores saturated during training.
- **Preemption resilience:** use `--append` to never lose harvested games on disconnect,
  and `--resume kepler64/training/gate_ckpt.npz` to continue training with full Adam momentum & RNG state.


In [ ]:
# Cell 3 — GPU smoke run with Self-Play Bootstrapping & Stockfish Mock Test
# Proves the end-to-end GPU path, deep self-play teacher, and benchmark in ~60 seconds.
!python scripts/credibility_gate.py --games 6 --max-plies 30 --steps 10 \
  --match-games 2 --match-move-ms 50 --seed 0 --batch-size 256 --workers 4 \
  --teacher-ms 500 \
  --benchmark-engine stockfish --benchmark-games 2 --benchmark-elo 1400 \
  --log-every 2 --ckpt-every 5 --allow-skew


In [ ]:
# Cell 4 — full scaled credibility gate with Sovereign Self-Play & Stockfish Mock Exam
# 1. Self-Play Harvest: Explorer (100ms) vs Deep Teacher (1000ms) generates physical move pairs.
# 2. Differentiable Training: 800 steps of Adam through the 17-parameter gravitational kernel.
# 3. Credibility Match: 200 games vs frozen baseline across 4 parallel workers.
# 4. Stockfish Mock Exam: 10 games vs Stockfish-1400 as an objective, non-contaminating audit.
!python scripts/credibility_gate.py --games 40 --max-plies 96 --steps 800 \
  --match-games 200 --match-move-ms 150 --seed 7 --batch-size 256 --workers 4 \
  --teacher-ms 1000 \
  --benchmark-engine stockfish --benchmark-games 10 --benchmark-elo 1400 \
  --append \
  --log-every 50 --ckpt-every 100 2>&1 | tee gate.log


In [ ]:
# Cell 5 — collect the claim (the only numbers allowed in public copy)
!tail -25 docs/credibility_gate_results.md
!ls -la docs/credibility_gate_results.* kepler64/training/gate_ckpt.npz kepler64/training/trained_constants_gate.json
